# Catálogo Unificado de Mídia: (Schema Merge)
* **The Movies Dataset + Goodreads Dataset**

In [19]:
# Configuração do Jupyter (Autoreload)
%load_ext autoreload
%autoreload 2

# Configuração de Caminho (Path Setup)
import sys
import os

# Adiciona a pasta raiz do projeto (..) ao sistema para liberar os imports locais
sys.path.append(os.path.abspath(os.path.join('..', '..')))

# Importação de Bibliotecas e Módulos
from sqlalchemy import create_engine
import pandas as pd

# Módulos customizados da pasta src/
import src.io.data_loader as dl
import src.io.data_save as ds
import src.io.db_client as dbc
import src.view.tables as tb

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


---

## Carregando Dados

In [20]:
# Dataset Books 
caminho = '../../data/processed/books/goodreads_books_master.parquet'
caminho_livros_generos = '../../data/processed/books/goodreads_books_genres_master.parquet'

df_books = dl.load_data(file_path=caminho, tipo_arquivo='parquet')
df_genres_books = dl.load_data(file_path=caminho_livros_generos, tipo_arquivo='parquet')

Dados Parquet carregados! Formato: (10897, 19)
Dados Parquet carregados! Formato: (94023, 19)


In [21]:
# Dataset Movies 
caminho = '../../data/processed/movies/movies_master.parquet'
caminho_filmes_generos = '../../data/processed/movies/movies_genres_master.parquet'

df_movies = dl.load_data(file_path=caminho, tipo_arquivo='parquet')
df_genres_movies = dl.load_data(file_path=caminho_filmes_generos, tipo_arquivo='parquet')

Dados Parquet carregados! Formato: (41091, 23)
Dados Parquet carregados! Formato: (86933, 23)


---
## Companrando Colunas

In [22]:
df_colunas_books = pd.DataFrame({
    "coluna": df_books.columns,
    "dtype_pandas": df_books.dtypes.values,
    "tipo_real": [
        df_books[col].map(type)[0].__name__
        if not df_books[col].dropna().empty else None
        for col in df_books.columns
    ]
})

display(tb.estilizar_tabela(
    df=df_colunas_books,
    caption="Colunas do dataframe"
))

,coluna,dtype_pandas,tipo_real
0,book_id,int64,int
1,title,object,str
2,author,object,str
3,average_rating,float64,float
4,isbn,object,str
5,original_language,object,str
6,num_pages,int64,int
7,total_votes,int64,int
8,text_reviews_count,int64,int
9,release_date,datetime64[ns],Timestamp


In [23]:
df_colunas_movies = pd.DataFrame({
    "coluna": df_movies.columns,
    "dtype_pandas": df_movies.dtypes.values,
    "tipo_real": [
        df_movies[col].map(type)[0].__name__
        if not df_movies[col].dropna().empty else None
        for col in df_movies.columns
    ]
})

display(tb.estilizar_tabela(
    df=df_colunas_movies,
    caption="Colunas do dataframe"
))

,coluna,dtype_pandas,tipo_real
0,belongs_to_collection,object,str
1,budget,float64,float
2,genres,object,ndarray
3,id,int64,int
4,original_language,object,str
5,original_title,object,str
6,popularity,float64,float
7,producer_company,object,ndarray
8,production_countries,object,ndarray
9,release_date,datetime64[ns],Timestamp


---
## Alinhamento de Esquemas

In [24]:
# Renomeando a chave de livros para o padrão universal
df_books.rename(columns={'book_id': 'id'}, inplace=True)
df_genres_books = df_genres_books.rename(columns={'book_id': 'id'})

In [25]:
# Conversão de Tipos Para Salvamento Futuro
df_books['producer_company'] = df_books['producer_company'].apply(lambda x: [x])
df_genres_books['producer_company'] = df_genres_books['producer_company'].apply(lambda x: [x])

---
## Carimbo de Domínio

In [26]:
# Criando o Carimbo de Domínio (A Nova Chave) 
df_books['media_type'] = 'Book'
df_movies['media_type'] = 'Movie'

df_genres_books['media_type'] = 'Book'
df_genres_movies['media_type'] = 'Movie'

---
## Empilhamento

In [27]:
# Schema Merge
df_catalog = pd.concat([df_movies, df_books], ignore_index=True)
df_genres_catalog = pd.concat([df_genres_movies, df_genres_books], ignore_index=True)

In [28]:
# Checagem de Integridade
print(f"Total de Filmes originais: {len(df_movies)}")
print(f"Total de Livros originais: {len(df_books)}")
print(f"Total no Catálogo Unificado: {len(df_catalog)}")
print(f"\nValidação: {len(df_movies) + len(df_books) == len(df_catalog)}")

Total de Filmes originais: 41091
Total de Livros originais: 10897
Total no Catálogo Unificado: 51988

Validação: True


In [29]:
display(tb.estilizar_tabela(
    df=df_catalog,
    qtd_linhas=10,
    caption="Catálogo Unifcado"
))

,belongs_to_collection,budget,genres,id,original_language,original_title,popularity,producer_company,production_countries,release_date,revenue,runtime,spoken_languages,title,average_rating,total_votes,global_score,release_year,decade,age_years,votes_per_year,popularity_tier,is_elite_engagement,media_type,author,isbn,num_pages,text_reviews_count
0,Toy Story Collection,"US$ 30,000,000",['Animation' 'Comedy' 'Family'],862,en,Toy Story,21.95,['Pixar Animation Studios'],['United States of America'],1995-10-30 00:00:00,"US$ 373,554,033",81 min,['English'],Toy Story,7.70,"5,415",77.00,1995,1990,31 anos,175,Mainstream Hit,True,Movie,nan,nan,nan,nan
1,Sem Coleção,"US$ 65,000,000",['Adventure' 'Fantasy' 'Family'],8844,en,Jumanji,17.02,['TriStar Pictures' 'Teitler Film' 'Interscope Communications'],['United States of America'],1995-12-15 00:00:00,"US$ 262,797,249",104 min,['English' 'Français'],Jumanji,6.90,"2,413",69.00,1995,1990,31 anos,78,Mainstream Hit,True,Movie,nan,nan,nan,nan
2,Grumpy Old Men Collection,nan,['Romance' 'Comedy'],15602,en,Grumpier Old Men,11.71,['Warner Bros.' 'Lancaster Gate'],['United States of America'],1995-12-22 00:00:00,nan,101 min,['English'],Grumpier Old Men,6.50,92,65.00,1995,1990,31 anos,3,Mainstream Hit,False,Movie,nan,nan,nan,nan
3,Sem Coleção,"US$ 16,000,000",['Comedy' 'Drama' 'Romance'],31357,en,Waiting to Exhale,3.86,['Twentieth Century Fox Film Corporation'],['United States of America'],1995-12-22 00:00:00,"US$ 81,452,156",127 min,['English'],Waiting to Exhale,6.10,34,61.00,1995,1990,31 anos,1,Mainstream Hit,False,Movie,nan,nan,nan,nan
4,Father of the Bride Collection,nan,['Comedy'],11862,en,Father of the Bride Part II,8.39,['Sandollar Productions' 'Touchstone Pictures'],['United States of America'],1995-02-10 00:00:00,"US$ 76,578,911",106 min,['English'],Father of the Bride Part II,5.70,173,57.00,1995,1990,31 anos,6,Mass Appeal,True,Movie,nan,nan,nan,nan
5,Sem Coleção,"US$ 60,000,000",['Action' 'Crime' 'Drama' 'Thriller'],949,en,Heat,17.92,['Regency Enterprises' 'Forward Pass' 'Warner Bros.'],['United States of America'],1995-12-15 00:00:00,"US$ 187,436,818",170 min,['English' 'Español'],Heat,7.70,"1,886",77.00,1995,1990,31 anos,61,Mainstream Hit,True,Movie,nan,nan,nan,nan
6,Sem Coleção,"US$ 58,000,000",['Comedy' 'Romance'],11860,en,Sabrina,6.68,['Paramount Pictures' 'Scott Rudin Productions' 'Mirage Enterprises' 'Sandollar Productions' 'Constellation Entertainment' 'Worldwide' 'Mont Blanc Entertainment GmbH'],['Germany' 'United States of America'],1995-12-15 00:00:00,nan,127 min,['Français' 'English'],Sabrina,6.20,141,62.00,1995,1990,31 anos,5,Mainstream Hit,True,Movie,nan,nan,nan,nan
7,Sem Coleção,nan,['Action' 'Adventure' 'Drama' 'Family'],45325,en,Tom and Huck,2.56,['Walt Disney Pictures'],['United States of America'],1995-12-22 00:00:00,nan,97 min,['English' 'Deutsch'],Tom and Huck,5.40,45,54.00,1995,1990,31 anos,1,Mass Appeal,False,Movie,nan,nan,nan,nan
8,Sem Coleção,"US$ 35,000,000",['Action' 'Adventure' 'Thriller'],9091,en,Sudden Death,5.23,['Universal Pictures' 'Imperial Entertainment' 'Signature Entertainment'],['United States of America'],1995-12-22 00:00:00,"US$ 64,350,171",106 min,['English'],Sudden Death,5.50,174,55.00,1995,1990,31 anos,6,Mass Appeal,True,Movie,nan,nan,nan,nan
9,James Bond Collection,"US$ 58,000,000",['Adventure' 'Action' 'Thriller'],710,en,GoldenEye,14.69,['United Artists' 'Eon Productions'],['United Kingdom' 'United States of America'],1995-11-16 00:00:00,"US$ 352,194,034",130 min,['English' 'Pусский' 'Español'],GoldenEye,6.60,"1,194",66.00,1995,1990,31 anos,39,Mainstream Hit,True,Movie,nan,nan,nan,nan


In [30]:
# Checagem de Integridade
print(f"Total de Filmes originais: {len(df_genres_movies)}")
print(f"Total de Livros originais: {len(df_genres_books)}")
print(f"Total no Catálogo Unificado: {len(df_genres_catalog)}")
print(f"\nValidação: {len(df_genres_movies) + len(df_genres_books) == len(df_genres_catalog)}")

Total de Filmes originais: 86933
Total de Livros originais: 94023
Total no Catálogo Unificado: 180956

Validação: True


In [31]:
display(tb.estilizar_tabela(
    df=df_genres_catalog,
    qtd_linhas=10,
    caption="Catálogo Unifcado"
))

,belongs_to_collection,budget,genres,id,original_language,original_title,popularity,producer_company,production_countries,release_date,revenue,runtime,spoken_languages,title,average_rating,total_votes,global_score,release_year,decade,age_years,votes_per_year,popularity_tier,is_elite_engagement,media_type,author,isbn,num_pages,text_reviews_count
0,Toy Story Collection,"US$ 30,000,000",Animation,862,en,Toy Story,21.95,['Pixar Animation Studios'],['United States of America'],1995-10-30 00:00:00,"US$ 373,554,033",81 min,['English'],Toy Story,7.70,"5,415",77.00,1995,1990,31 anos,175,Mainstream Hit,True,Movie,nan,nan,nan,nan
1,Toy Story Collection,"US$ 30,000,000",Comedy,862,en,Toy Story,21.95,['Pixar Animation Studios'],['United States of America'],1995-10-30 00:00:00,"US$ 373,554,033",81 min,['English'],Toy Story,7.70,"5,415",77.00,1995,1990,31 anos,175,Mainstream Hit,True,Movie,nan,nan,nan,nan
2,Toy Story Collection,"US$ 30,000,000",Family,862,en,Toy Story,21.95,['Pixar Animation Studios'],['United States of America'],1995-10-30 00:00:00,"US$ 373,554,033",81 min,['English'],Toy Story,7.70,"5,415",77.00,1995,1990,31 anos,175,Mainstream Hit,True,Movie,nan,nan,nan,nan
3,Sem Coleção,"US$ 65,000,000",Adventure,8844,en,Jumanji,17.02,['TriStar Pictures' 'Teitler Film' 'Interscope Communications'],['United States of America'],1995-12-15 00:00:00,"US$ 262,797,249",104 min,['English' 'Français'],Jumanji,6.90,"2,413",69.00,1995,1990,31 anos,78,Mainstream Hit,True,Movie,nan,nan,nan,nan
4,Sem Coleção,"US$ 65,000,000",Fantasy,8844,en,Jumanji,17.02,['TriStar Pictures' 'Teitler Film' 'Interscope Communications'],['United States of America'],1995-12-15 00:00:00,"US$ 262,797,249",104 min,['English' 'Français'],Jumanji,6.90,"2,413",69.00,1995,1990,31 anos,78,Mainstream Hit,True,Movie,nan,nan,nan,nan
5,Sem Coleção,"US$ 65,000,000",Family,8844,en,Jumanji,17.02,['TriStar Pictures' 'Teitler Film' 'Interscope Communications'],['United States of America'],1995-12-15 00:00:00,"US$ 262,797,249",104 min,['English' 'Français'],Jumanji,6.90,"2,413",69.00,1995,1990,31 anos,78,Mainstream Hit,True,Movie,nan,nan,nan,nan
6,Grumpy Old Men Collection,nan,Romance,15602,en,Grumpier Old Men,11.71,['Warner Bros.' 'Lancaster Gate'],['United States of America'],1995-12-22 00:00:00,nan,101 min,['English'],Grumpier Old Men,6.50,92,65.00,1995,1990,31 anos,3,Mainstream Hit,False,Movie,nan,nan,nan,nan
7,Grumpy Old Men Collection,nan,Comedy,15602,en,Grumpier Old Men,11.71,['Warner Bros.' 'Lancaster Gate'],['United States of America'],1995-12-22 00:00:00,nan,101 min,['English'],Grumpier Old Men,6.50,92,65.00,1995,1990,31 anos,3,Mainstream Hit,False,Movie,nan,nan,nan,nan
8,Sem Coleção,"US$ 16,000,000",Comedy,31357,en,Waiting to Exhale,3.86,['Twentieth Century Fox Film Corporation'],['United States of America'],1995-12-22 00:00:00,"US$ 81,452,156",127 min,['English'],Waiting to Exhale,6.10,34,61.00,1995,1990,31 anos,1,Mainstream Hit,False,Movie,nan,nan,nan,nan
9,Sem Coleção,"US$ 16,000,000",Drama,31357,en,Waiting to Exhale,3.86,['Twentieth Century Fox Film Corporation'],['United States of America'],1995-12-22 00:00:00,"US$ 81,452,156",127 min,['English'],Waiting to Exhale,6.10,34,61.00,1995,1990,31 anos,1,Mainstream Hit,False,Movie,nan,nan,nan,nan


---
## Salvando Dataset

In [32]:
# Salvando DataFrame 
ds.save_dataset(
    df=df_catalog,
    pasta='../../data/processed',
    nome_arquivo='unified_media_catalog.parquet', 
    tipo_arquivo='parquet'
)

Sucesso! Ficheiro guardado em '..\..\data\processed\unified_media_catalog.parquet.parquet'


#### Salvando como Banco de Dados

In [33]:
dbc.save_db(
    df=df_catalog,
    pasta='../../data/processed',
    nome_banco='unified_media_catalog',
    nome_tabela='unified_media_catalog'
)


Colunas achatadas para string: ['genres', 'producer_company', 'production_countries', 'spoken_languages']

Sucesso! Tabela 'unified_media_catalog' criada no banco: 'unified_media_catalog.db'


In [34]:
dbc.save_db(
    df=df_genres_catalog,
    pasta='../../data/processed',
    nome_banco='unified_media_catalog',
    nome_tabela='unified_media_genres' 
)


Colunas achatadas para string: ['producer_company', 'production_countries', 'spoken_languages']

Sucesso! Tabela 'unified_media_genres' criada no banco: 'unified_media_catalog.db'


In [35]:
# Execução da Query de Auditoria
caminho_banco = '../../data/processed/unified_media_catalog.db'

query_auditoria = """
    SELECT 
        media_type, 
        COUNT(*) as total_registros 
    FROM unified_media_catalog 
    GROUP BY media_type;
"""

df_auditoria = dbc.execute_query(
    caminho_db=caminho_banco,
    alvo=query_auditoria,
    eh_query=True
)

# Verificação de segurança (Garantindo que o retorno do banco é válido)
if df_auditoria is not None:
    linhas_originais = len(df_catalog)
    linhas_banco = df_auditoria['total_registros'].sum()

    if linhas_originais == linhas_banco:
        print(f"\nSucesso absoluto! Todas as {linhas_originais} linhas foram transferidas com perfeição.")
        display(tb.estilizar_tabela(df=df_auditoria))
    else:
        print(f"\nAlerta: Discrepância. {linhas_originais} no Pandas vs {linhas_banco} no Banco.")
else:
    print("\nErro: Não foi possível realizar a verificação de segurança pois a consulta falhou.")


Dados carregados via SQL! Formato: (2, 2)

Sucesso absoluto! Todas as 51988 linhas foram transferidas com perfeição.


,media_type,total_registros
0,Book,10897
1,Movie,41091


In [36]:
# Execução da Query de Auditoria
caminho_banco = '../../data/processed/unified_media_catalog.db'

query_auditoria = """
    SELECT 
        media_type, 
        COUNT(*) as total_registros 
    FROM unified_media_genres 
    GROUP BY media_type;
"""

df_auditoria = dbc.execute_query(
    caminho_db=caminho_banco,
    alvo=query_auditoria,
    eh_query=True
)

# Verificação de segurança (Garantindo que o retorno do banco é válido)
if df_auditoria is not None:
    linhas_originais = len(df_genres_catalog)
    linhas_banco = df_auditoria['total_registros'].sum()

    if linhas_originais == linhas_banco:
        print(f"\nSucesso absoluto! Todas as {linhas_originais} linhas foram transferidas com perfeição.")
        display(tb.estilizar_tabela(df=df_auditoria))
    else:
        print(f"\nAlerta: Discrepância. {linhas_originais} no Pandas vs {linhas_banco} no Banco.")
else:
    print("\nErro: Não foi possível realizar a verificação de segurança pois a consulta falhou.")


Dados carregados via SQL! Formato: (2, 2)

Sucesso absoluto! Todas as 180956 linhas foram transferidas com perfeição.


,media_type,total_registros
0,Book,94023
1,Movie,86933


#  Data Integration: Unified Media Catalog (Schema Merge)

Nesta etapa crucial do projeto, avançamos da higienização de domínios isolados para a construção da nossa **Camada Ouro (Gold Layer)**. O objetivo aqui é unificar os ecossistemas de Literatura e Cinema em uma única Tabela Fato (*Wide Table*), consolidando as métricas universais de entretenimento sem perder a riqueza das variáveis específicas de cada mídia.

### Roteiro de Arquitetura e Integração:

1. **Schema Alignment (Alinhamento de Esquemas):**
   * Padronização das chaves primárias de ambos os datasets para a nomenclatura universal (`id`).
   * Envelopamento da coluna `producer_company` de livros em estruturas de lista (`list`), garantindo a compatibilidade de tipos com o formato do dataset de filmes e evitando rejeição na serialização do Parquet.

2. **Domain Stamping (Carimbo de Domínio):**
   * Criação da feature obrigatória de negócio `media_type` (`'Movie'` ou `'Book'`), que servirá como o filtro principal do nosso catálogo unificado.

3. **Data Concatenation (Empilhamento / Wide Table):**
   * Integração dos datasets via `pd.concat`.
   * **Colunas Universais:** Alinhamento perfeito de métricas compartilhadas (ex: `title`, `total_votes`, `release_year`, `is_elite_engagement`).
   * **Colunas Específicas:** Preservação integral de atributos de nicho (ex: `budget`, `isbn`, `num_pages`), utilizando `NaN` de forma controlada onde o contexto não se aplica.

4. **Persistência e Arquitetura de Consumo:**
   Para garantir a flexibilidade da nossa **Serving Layer (Camada de Consumo)**, os dados consolidados foram exportados em duas frentes estratégicas:
   * **Camada de Arquivo (Parquet):** O catálogo foi salvo na raiz do diretório de consumo (`../../data/processed/unified_media_catalog.parquet`). Este formato colunar foi escolhido por sua alta taxa de compressão e tipagem estrita, sendo a fonte ideal para ferramentas de visualização e BI (como o Power BI).
   * **Camada Analítica (Banco de Dados Relacional):** Após o achatamento (serialização) de arrays estruturados em texto puro, o catálogo foi ingerido em um banco de dados SQLite (`../../data/processed/unified_media_catalog.db`). O uso de um SGBD nesta etapa encerra o ciclo de Engenharia de Dados (Python/Pandas) e estabelece uma *Single Source of Truth* (Fonte Única da Verdade) para a Fase de Analytics, permitindo a extração de insights de negócio através de consultas SQL avançadas.